In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score, KFold
from lightgbm import LGBMRegressor
import shap

train = pd.read_csv('../data/raw/train.csv')
test = pd.read_csv('../data/raw/test.csv')
print(f"Train: {train.shape}, Test: {test.shape}")

In [ ]:
train = train[~((train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000))]
 
y_train = np.log1p(train['SalePrice'])
X_train = train.drop(['SalePrice', 'Id'], axis=1)

print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")

In [ ]:
test_ids = test['Id'] 
X_test = test.drop(['Id'], axis=1)

print(f"X_test: {X_test.shape}")

In [ ]:
n_train = len(X_train)

combined = pd.concat([X_train, X_test], axis=0, ignore_index=True)

combined['TotalSF'] = combined['1stFlrSF'] + combined['2ndFlrSF'] + combined['TotalBsmtSF']

numeric_cols = combined.select_dtypes(include=[np.number]).columns
combined[numeric_cols] = combined[numeric_cols].fillna(combined[numeric_cols].median())

cat_cols = combined.select_dtypes(include=['object']).columns
combined[cat_cols] = combined[cat_cols].fillna("Missing")

combined = pd.get_dummies(combined, drop_first=True)

X_train_final = combined.iloc[:n_train]
X_test_final = combined.iloc[n_train:]

print(f"X_train_final: {X_train_final.shape}")
print(f"X_test_final: {X_test_final.shape}")
print(f"NaN count: {combined.isnull().sum().sum()}")

In [ ]:
best_params = {
    'n_estimators': 554, 
    'learning_rate': 0.02412519173026985, 
    'max_depth': 3, 
    'num_leaves': 117, 
    'min_child_samples': 21, 
    'reg_alpha': 0.03773710925681085, 
    'reg_lambda': 9.634562038742856e-07, 
    'subsample': 0.9984756772826762, 
    'colsample_bytree': 0.6144169597126988,
    'random_state': 42,
    'verbosity': -1,
    'n_jobs': -1,
}

final_model = LGBMRegressor(**best_params)
final_model.fit(X_train_final, y_train)

In [ ]:
explainer = shap.Explainer(final_model)
shap_values = explainer(X_train_final)
print("Shape: ", shap_values.values.shape)
print(f"Base value (log-space): {shap_values.base_values[0]:.4f}")
print(f"Base value ($): ${np.expm1(shap_values.base_values[0]):,.0f}")

In [ ]:
shap.plots.beeswarm(shap_values, max_display=15, show=False)
plt.tight_layout()
plt.savefig('../reports/figures/05_shap_summary.png', dpi=120, bbox_inches='tight')
plt.show()

## Summary

This SHAP Beeswarm plot shows which features have the biggest impact on the model predictions. And it shows whether feature values increase or decrease the predicted house price.

Beeswarm automatically shows descending values, from the highest impact to the lowest

Top 3 Features:
- TotalSF (0.110)
- OverallQual (0.0962)
- YearRemodAdd (0.0318)

TotalSF, which we engineered in 03_feature_engineering.ipynb, has the highest impact

In [ ]:
preds_log = final_model.predict(X_test_final)

preds = np.expm1(preds_log)

submission = pd.DataFrame({
    'Id': test_ids,
    'SalePrice': preds
})

submission.to_csv('../data/processed/submission.csv', index=False)
print(f"Submission shape: {submission.shape}")
print(submission.head())

## Summary

In this final notebook, I trained the optimized LightGBM model with the best Optuna hyperparameters on the full training dataset and generated predictions for the Kaggle test set

The final Kaggle submission achieved **0.12753 RMSLE**, placing **1409 / 5119 teams (top 27.5%)** on the public leaderboard

The Kaggle score was slightly worse than the local cross-validation score (0.1190 RMSE), but both results were consistent and predicted well